
# 同一模型在不同情绪类别上的准确性差异评估

本实验固定使用同一个英文 6 类情绪分类模型：

- 模型：`bhadresh-savani/distilbert-base-uncased-emotion`
- 数据集：`dair-ai/emotion`
- 测试类别：`sadness / joy / love / anger / fear / surprise`

主要输出：

1. 总体 Accuracy、Balanced Accuracy、Macro-F1、Weighted-F1
2. 每类“分类准确率”（这里定义为该类别的 Recall，即该类样本中被正确识别的比例）
3. 每类 Precision、F1、样本数和 95% Wilson 置信区间
4. 各类别准确率最大差距
5. 混淆矩阵与主要误判方向
6. 高置信度误判样本
7. 卡方检验：不同情绪类别的正确率是否存在统计差异

> 注意：该模型本身是在这一情绪数据集上微调的。因此，这是一项**同分布、留出测试集评估**，适合检查模型对各类别是否均衡，但不能直接代表模型在新闻、客服、医疗或中文文本等新领域的泛化能力。


In [ ]:

# 安装依赖。首次运行后如出现版本提示，通常无需重启运行时。
!pip -q install -U transformers datasets accelerate scikit-learn scipy pandas matplotlib


In [ ]:

import random
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from scipy.stats import chi2_contingency
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from IPython.display import display

SEED = 42
MODEL_NAME = "bhadresh-savani/distilbert-base-uncased-emotion"
DATASET_NAME = "dair-ai/emotion"
DATASET_CONFIG = "split"
SPLIT = "test"

BATCH_SIZE = 64
MAX_LENGTH = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("运行设备：", device)



## 1. 加载测试集与模型

代码会从模型配置和数据集特征中读取标签名称，并检查二者是否一致，避免把不同编号的标签错误对应。


In [ ]:

dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
test_ds = dataset[SPLIT]

dataset_label_names = list(test_ds.features["label"].names)
dataset_label_to_id = {name.lower(): i for i, name in enumerate(dataset_label_names)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

def normalize_label(label):
    return str(label).strip().lower().replace(" ", "_")

model_id2label = {
    int(k): normalize_label(v)
    for k, v in model.config.id2label.items()
}

# 一些模型配置可能只写 LABEL_0、LABEL_1……
# 只有在类别数完全一致时，才按数据集编号回退。
generic_labels = all(v.startswith("label_") for v in model_id2label.values())
if generic_labels:
    if model.config.num_labels != len(dataset_label_names):
        raise ValueError("模型类别数与数据集类别数不一致，不能安全映射。")
    warnings.warn("模型配置使用通用 LABEL_n，按数据集标签编号进行映射。")
    model_id2label = {
        i: normalize_label(name)
        for i, name in enumerate(dataset_label_names)
    }

model_label_set = set(model_id2label.values())
dataset_label_set = set(dataset_label_to_id.keys())

if model_label_set != dataset_label_set:
    raise ValueError(
        "模型和数据集标签集合不一致。\n"
        f"模型标签：{sorted(model_label_set)}\n"
        f"数据集标签：{sorted(dataset_label_set)}"
    )

print("测试样本数：", len(test_ds))
print("数据集标签：", dataset_label_names)
print("模型标签映射：", model_id2label)



## 2. 批量推理

保存每条文本的真实标签、预测标签、最大预测概率和是否预测正确。


In [ ]:

texts = test_ds["text"]
y_true = np.asarray(test_ds["label"], dtype=int)

pred_model_ids = []
confidences = []

for start in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[start : start + BATCH_SIZE]
    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)
        batch_conf, batch_pred = probs.max(dim=-1)

    pred_model_ids.extend(batch_pred.cpu().numpy().tolist())
    confidences.extend(batch_conf.cpu().numpy().tolist())

# 通过标签名称把模型编号转换成数据集编号，避免默认假设编号顺序一致。
pred_label_names = [model_id2label[int(i)] for i in pred_model_ids]
y_pred = np.asarray(
    [dataset_label_to_id[name] for name in pred_label_names],
    dtype=int,
)
confidences = np.asarray(confidences, dtype=float)

ZH_LABELS = {
    "sadness": "悲伤",
    "joy": "喜悦",
    "love": "爱/喜爱",
    "anger": "愤怒",
    "fear": "恐惧",
    "surprise": "惊讶",
}

results_df = pd.DataFrame({
    "text": texts,
    "true_id": y_true,
    "pred_id": y_pred,
    "true_label": [dataset_label_names[i] for i in y_true],
    "pred_label": [dataset_label_names[i] for i in y_pred],
    "confidence": confidences,
})
results_df["correct"] = results_df["true_id"] == results_df["pred_id"]

display(results_df.head())



## 3. 总体指标

- **Accuracy**：所有测试样本中预测正确的比例。
- **Balanced Accuracy**：各类别 Recall 的平均值，不会让样本较多的类别占据更大权重。
- **Macro-F1**：先分别计算各类 F1，再做简单平均。
- **Weighted-F1**：按各类别样本量加权。


In [ ]:

overall_metrics = pd.DataFrame([{
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(y_true, y_pred, average="macro"),
    "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    "n_test": len(y_true),
}])

display(overall_metrics.style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
}))



## 4. 每一种情绪的分类准确性

这里把“某类准确率”定义为：

\[
\text{Class Accuracy}_i
= \frac{\text{该类预测正确数}}{\text{该类真实样本总数}}
= \text{Recall}_i
\]

这比 one-vs-rest accuracy 更适合比较不同情绪，因为后者包含大量“非该类”真负例，数值往往虚高。

95% 置信区间使用 Wilson score interval。样本越少，区间通常越宽。


In [ ]:

def wilson_interval(correct, total, z=1.96):
    # 二项比例的 Wilson 95% 置信区间
    if total == 0:
        return np.nan, np.nan
    p = correct / total
    denominator = 1 + (z**2 / total)
    center = (p + z**2 / (2 * total)) / denominator
    margin = (
        z
        * np.sqrt((p * (1 - p) / total) + (z**2 / (4 * total**2)))
        / denominator
    )
    return max(0.0, center - margin), min(1.0, center + margin)

label_ids = list(range(len(dataset_label_names)))
cm = confusion_matrix(y_true, y_pred, labels=label_ids)
report = classification_report(
    y_true,
    y_pred,
    labels=label_ids,
    target_names=dataset_label_names,
    output_dict=True,
    zero_division=0,
)

rows = []
n_total = len(y_true)

for i, label in enumerate(dataset_label_names):
    tp = int(cm[i, i])
    support = int(cm[i, :].sum())
    predicted_as_class = int(cm[:, i].sum())
    fn = support - tp
    fp = predicted_as_class - tp
    tn = n_total - tp - fn - fp

    class_accuracy = tp / support if support else np.nan
    ovr_accuracy = (tp + tn) / n_total if n_total else np.nan
    ci_low, ci_high = wilson_interval(tp, support)

    class_rows = results_df[results_df["true_id"] == i]
    correct_rows = class_rows[class_rows["correct"]]
    incorrect_rows = class_rows[~class_rows["correct"]]

    rows.append({
        "emotion": label,
        "emotion_zh": ZH_LABELS.get(label, label),
        "support": support,
        "correct": tp,
        "incorrect": fn,
        "class_accuracy_recall": class_accuracy,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "precision": report[label]["precision"],
        "f1": report[label]["f1-score"],
        "one_vs_rest_accuracy": ovr_accuracy,
        "mean_confidence_all": class_rows["confidence"].mean(),
        "mean_confidence_correct": correct_rows["confidence"].mean(),
        "mean_confidence_wrong": incorrect_rows["confidence"].mean(),
    })

per_class_df = (
    pd.DataFrame(rows)
    .sort_values("class_accuracy_recall", ascending=False)
    .reset_index(drop=True)
)

display(per_class_df.style.format({
    "class_accuracy_recall": "{:.2%}",
    "ci95_low": "{:.2%}",
    "ci95_high": "{:.2%}",
    "precision": "{:.2%}",
    "f1": "{:.2%}",
    "one_vs_rest_accuracy": "{:.2%}",
    "mean_confidence_all": "{:.3f}",
    "mean_confidence_correct": "{:.3f}",
    "mean_confidence_wrong": "{:.3f}",
}))



## 5. 自动总结类别差异

输出表现最好、最差的情绪，以及二者相差多少个百分点。标准差反映各类 Recall 的离散程度。


In [ ]:

best_row = per_class_df.iloc[0]
worst_row = per_class_df.iloc[-1]

accuracy_gap = (
    best_row["class_accuracy_recall"]
    - worst_row["class_accuracy_recall"]
)
class_accuracy_std = per_class_df["class_accuracy_recall"].std(ddof=0)

summary_df = pd.DataFrame([{
    "best_emotion": f'{best_row["emotion"]}（{best_row["emotion_zh"]}）',
    "best_accuracy": best_row["class_accuracy_recall"],
    "worst_emotion": f'{worst_row["emotion"]}（{worst_row["emotion_zh"]}）',
    "worst_accuracy": worst_row["class_accuracy_recall"],
    "max_min_gap": accuracy_gap,
    "class_accuracy_std": class_accuracy_std,
}])

display(summary_df.style.format({
    "best_accuracy": "{:.2%}",
    "worst_accuracy": "{:.2%}",
    "max_min_gap": "{:.2%}",
    "class_accuracy_std": "{:.2%}",
}))

print(
    f'表现最好：{best_row["emotion"]}（{best_row["emotion_zh"]}），'
    f'{best_row["class_accuracy_recall"]:.2%}\n'
    f'表现最差：{worst_row["emotion"]}（{worst_row["emotion_zh"]}），'
    f'{worst_row["class_accuracy_recall"]:.2%}\n'
    f'最大差距：{accuracy_gap * 100:.2f} 个百分点'
)



## 6. 检验不同类别正确率是否存在统计差异

建立“情绪类别 × 是否预测正确”的列联表，并进行卡方独立性检验。

- \(p < 0.05\)：至少部分情绪类别的正确率存在统计差异。
- \(p \ge 0.05\)：现有测试样本不足以证明类别正确率不同。
- Cramér's V 是效应量；越接近 0，类别和预测正确与否的关联越弱。

卡方检验只说明总体上存在差异，不说明具体是哪两类不同。


In [ ]:

correct_incorrect_table = np.column_stack([
    np.diag(cm),
    cm.sum(axis=1) - np.diag(cm),
])

chi2, p_value, dof, expected = chi2_contingency(correct_incorrect_table)
n = correct_incorrect_table.sum()
min_dim = min(correct_incorrect_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

significance_df = pd.DataFrame([{
    "chi_square": chi2,
    "degrees_of_freedom": dof,
    "p_value": p_value,
    "cramers_v": cramers_v,
    "significant_at_0.05": p_value < 0.05,
}])

display(significance_df.style.format({
    "chi_square": "{:.4f}",
    "p_value": "{:.6g}",
    "cramers_v": "{:.4f}",
}))

if p_value < 0.05:
    print("结论：不同情绪类别的分类正确率存在统计显著差异（p < 0.05）。")
else:
    print("结论：当前测试集未显示出统计显著的类别正确率差异（p ≥ 0.05）。")


## 7. 每类准确率及 95% 置信区间

In [ ]:

plot_df = per_class_df.sort_values("class_accuracy_recall", ascending=False).copy()

x = np.arange(len(plot_df))
values = plot_df["class_accuracy_recall"].to_numpy()
lower_errors = values - plot_df["ci95_low"].to_numpy()
upper_errors = plot_df["ci95_high"].to_numpy() - values

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, values)
ax.errorbar(
    x,
    values,
    yerr=np.vstack([lower_errors, upper_errors]),
    fmt="none",
    capsize=4,
)
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{e}\n({z})' for e, z in zip(plot_df["emotion"], plot_df["emotion_zh"])],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Class accuracy / Recall")
ax.set_title("Per-emotion classification accuracy with 95% Wilson CI")
ax.grid(axis="y", alpha=0.3)

for i, value in enumerate(values):
    ax.text(
        i,
        min(value + upper_errors[i] + 0.025, 1.03),
        f"{value:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig("per_emotion_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()



## 8. 归一化混淆矩阵

每一行代表真实类别，行内各格表示该真实类别被预测成各类别的比例。  
对角线越高越好；非对角线中的大值表示主要混淆方向。


In [ ]:

row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0,
)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm_normalized, aspect="auto")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(np.arange(len(dataset_label_names)))
ax.set_yticks(np.arange(len(dataset_label_names)))
ax.set_xticklabels(dataset_label_names, rotation=45, ha="right")
ax.set_yticklabels(dataset_label_names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Row-normalized confusion matrix")

threshold = cm_normalized.max() / 2
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        value = cm_normalized[i, j]
        ax.text(
            j,
            i,
            f"{value:.1%}\n(n={cm[i, j]})",
            ha="center",
            va="center",
            color="white" if value > threshold else "black",
            fontsize=8,
        )

plt.tight_layout()
plt.savefig("normalized_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()



## 9. 查看主要误判方向和高置信度错误

高置信度错误通常比普通错误更值得检查，因为它可能提示：

- 标签定义存在重叠；
- 文本包含多个情绪；
- 讽刺、否定或上下文缺失；
- 数据标注可能有争议；
- 模型学到了特定词语捷径。


In [ ]:

error_df = results_df[~results_df["correct"]].copy()

error_pairs = (
    error_df
    .groupby(["true_label", "pred_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("最常见的误判方向：")
display(error_pairs.head(20))

high_confidence_errors = (
    error_df
    .sort_values("confidence", ascending=False)
    [["text", "true_label", "pred_label", "confidence"]]
    .head(30)
)

print("置信度最高的误判样本：")
display(high_confidence_errors)


## 10. 分类别抽取高置信度误判样本

In [ ]:

examples_per_class = 5

for label in dataset_label_names:
    subset = (
        error_df[error_df["true_label"] == label]
        .sort_values("confidence", ascending=False)
        [["text", "true_label", "pred_label", "confidence"]]
        .head(examples_per_class)
    )
    print(f"\n真实类别：{label}（{ZH_LABELS.get(label, label)}）")
    display(subset)



## 11. 导出结果

会保存：

- `overall_metrics.csv`
- `per_class_metrics.csv`
- `all_predictions.csv`
- `error_pairs.csv`
- `high_confidence_errors.csv`
- `per_emotion_accuracy.png`
- `normalized_confusion_matrix.png`


In [ ]:

overall_metrics.to_csv("overall_metrics.csv", index=False)
per_class_df.to_csv("per_class_metrics.csv", index=False)
results_df.to_csv("all_predictions.csv", index=False)
error_pairs.to_csv("error_pairs.csv", index=False)
high_confidence_errors.to_csv("high_confidence_errors.csv", index=False)

print("文件已保存到当前 Colab 运行目录。")

# 在 Colab 中按需取消注释，下载单个文件：
# from google.colab import files
# files.download("per_class_metrics.csv")



## 如何解读实验结果

建议重点关注以下几项：

1. **Class accuracy / Recall**  
   这是比较模型对每种真实情绪识别能力的主要指标。

2. **95% 置信区间**  
   两类点估计不同，并不一定意味着真实能力存在明显差异。样本较少的类别区间往往更宽。

3. **Precision 与 Recall 的组合**  
   - Recall 低：该情绪容易漏检，被误判成其他类。
   - Precision 低：模型经常把其他情绪误报成该类。

4. **混淆矩阵**  
   用于定位具体类别对，例如 `love → joy`、`fear → surprise`。

5. **总体 Accuracy 与 Balanced Accuracy 的差距**  
   若二者相差较大，说明类别不均衡或模型对不同类别的表现不均衡。

6. **高置信度误判**  
   应人工阅读，判断错误来自模型、文本歧义还是标签质量。

### 报告文字模板

> 在 `dair-ai/emotion` 官方测试集上，我们使用同一 DistilBERT 情绪分类模型评估六种情绪的分类性能。总体准确率为 `{运行结果}`，宏平均 F1 为 `{运行结果}`。以各类别 Recall 作为类别准确率时，表现最佳的类别为 `{运行结果}`，表现最弱的类别为 `{运行结果}`，二者相差 `{运行结果}` 个百分点。归一化混淆矩阵显示，主要误判发生在 `{运行结果}` 之间。卡方检验结果为 `p={运行结果}`，表明不同情绪类别的正确率 `{存在/不存在}` 统计显著差异。由于模型在同源数据上微调，本结果主要反映类间识别均衡性，不等同于跨领域泛化能力。
